<link rel="stylesheet" href="/site-assets/css/gemma.css">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Google+Symbols:opsz,wght,FILL,GRAD@20..48,100..700,0..1,-50..200" />

##### Copyright 2024 Google LLC.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Fine-tune Gemma models in Keras using LoRA


<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://ai.google.dev/gemma/docs/lora_tuning"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />View on ai.google.dev</a>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/google/generative-ai-docs/blob/main/site/en/gemma/docs/lora_tuning.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/google/generative-ai-docs/main/site/en/gemma/docs/lora_tuning.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />Open in Vertex AI</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/google/generative-ai-docs/blob/main/site/en/gemma/docs/lora_tuning.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
  </td>
</table>

## Overview

Gemma is a family of lightweight, state-of-the art open models built from the same research and technology used to create the Gemini models.

Large Language Models (LLMs) like Gemma have been shown to be effective at a variety of NLP tasks. An LLM is first pre-trained on a large corpus of text in a self-supervised fashion. Pre-training helps LLMs learn general-purpose knowledge, such as statistical relationships between words. An LLM can then be fine-tuned with domain-specific data to perform downstream tasks (such as sentiment analysis).

LLMs are extremely large in size (parameters in the order of billions). Full fine-tuning (which updates all the parameters in the model) is not required for most applications because typical fine-tuning datasets are relatively much smaller than the pre-training datasets.

[Low Rank Adaptation (LoRA)](https://arxiv.org/abs/2106.09685) is a fine-tuning technique which greatly reduces the number of trainable parameters for downstream tasks by freezing the weights of the model and inserting a smaller number of new weights into the model. This makes training with LoRA much faster and more memory-efficient, and produces smaller model weights (a few hundred MBs), all while maintaining the quality of the model outputs.

This tutorial walks you through using KerasNLP to perform LoRA fine-tuning on a Gemma 2B model using the [Databricks Dolly 15k dataset](https://huggingface.co/datasets/databricks/databricks-dolly-15k). This dataset contains 15,000 high-quality human-generated prompt / response pairs specifically designed for fine-tuning LLMs.

## Setup

### Get access to Gemma

To complete this tutorial, you will first need to complete the setup instructions at [Gemma setup](https://ai.google.dev/gemma/docs/setup). The Gemma setup instructions show you how to do the following:

* Get access to Gemma on [kaggle.com](https://kaggle.com).
* Select a Colab runtime with sufficient resources to run
  the Gemma 2B model.
* Generate and configure a Kaggle username and API key.

After you've completed the Gemma setup, move on to the next section, where you'll set environment variables for your Colab environment.

### Select the runtime

To complete this tutorial, you'll need to have a Colab runtime with sufficient resources to run the Gemma model. In this case, you can use a T4 GPU:

1. In the upper-right of the Colab window, select &#9662; (**Additional connection options**).
2. Select **Change runtime type**.
3. Under **Hardware accelerator**, select **T4 GPU**.

### Configure your API key

To use Gemma, you must provide your Kaggle username and a Kaggle API key.

To generate a Kaggle API key, go to the **Account** tab of your Kaggle user profile and select **Create New Token**. This will trigger the download of a `kaggle.json` file containing your API credentials.

In Colab, select **Secrets** (🔑) in the left pane and add your Kaggle username and Kaggle API key. Store your username under the name `KAGGLE_USERNAME` and your API key under the name `KAGGLE_KEY`.

### Set environment variables

Set environment variables for `KAGGLE_USERNAME` and `KAGGLE_KEY`.

In [ ]:
import os
from google.colab import userdata

# Note: `userdata.get` is a Colab API. If you're not using Colab, set the env
# vars as appropriate for your system.

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

### Install dependencies

Install Keras, KerasNLP, and other dependencies.

In [ ]:
# Install Keras 3 last. See https://keras.io/getting_started/ for more details.
# !pip install tensorflow==2.17.0

!pip install -q -U keras-nlp
!pip install -q -U "keras>=3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 691.2/691.2 kB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 106.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.3/615.3 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 102.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.17.0 requires tensorflow<2.18,>=2.17, but you have tensorflow 2.18.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.17.0 requires tensorflow<2.18,>=2.17, but you have tensorflow 2.18.0 which is incompatible.


In [ ]:
!pip install -U peft

### Select a backend

Keras is a high-level, multi-framework deep learning API designed for simplicity and ease of use. Using Keras 3, you can run workflows on one of three backends: TensorFlow, JAX, or PyTorch.

For this tutorial, configure the backend for JAX.

In [ ]:
os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow".
# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

In [ ]:
# import os

# # Set the backend to JAX
# os.environ["KERAS_BACKEND"] = "jax"

# # Adjust memory allocation settings
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.80"  # Use only 80% of GPU memory
# os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"  # Allow dynamic memory allocation

# # Import KerasNLP after setting environment variables
# import keras_nlp

### Import packages

Import Keras and KerasNLP.

In [ ]:
import keras
import keras_nlp

## Load Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


In [ ]:
import pandas as pd

csv_path = '/content/gdrive/MyDrive/DATI/illuminismo/illuminismo_dataset_prima_parte_merged_20240112_no_source .csv'
df = pd.read_csv(csv_path)

df.head()

,completion,generated_prompt
0,Forse potran col tempo sembrar troppo animosi ...,Descrivi il pericolo della pedanteria nel camp...
1,quel posporre e disprezzare che si fa da alcun...,Quali sono gli effetti del culto delle parole ...
2,È ridicola cosa il raccomandarsi alla benevole...,Cosa significa meritarsi l'approvazione del pu...
3,Ciò che è piaciuto diviso in fogli conviene sp...,Quali sono le caratteristiche stilistiche e re...
4,E sin a quando fate voi conto di continuare qu...,Quali sono le intenzioni future dietro il pros...


In [ ]:
# !wget -O databricks-dolly-15k.jsonl https://huggingface.co/datasets/databricks/databricks-dolly-15k/resolve/main/databricks-dolly-15k.jsonl

Preprocess the data. This tutorial uses a subset of 1000 training examples to execute the notebook faster. Consider using more training data for higher quality fine-tuning.

In [ ]:
# import json
# data = []
# with open("databricks-dolly-15k.jsonl") as file:
#     for line in file:
#         features = json.loads(line)
#         # Filter out examples with context, to keep it simple.
#         if features["context"]:
#             continue
#         # Format the entire example as a single string.
#         template = "Instruction:\n{instruction}\n\nResponse:\n{response}"
#         data.append(template.format(**features))

# # Only use 1000 training examples, to keep it fast.
# data = data[:1000]

In [ ]:
# data = []

# # df = df[:100]

# # Define the template

# # Crea una nuova colonna con il template formattato
# df["formatted"] = df.apply(
#     lambda row: f"Instruction:\n{row['generated_prompt']}\n\nResponse:\n{row['completion']}",
#     axis=1
# )

In [ ]:
data = []

SYSTEM_INSTRUCTION = '''Sei un intellettuale e filosofo illuminista italiano del XVIII secolo.
Rispondi alle domande utilizzando lo stile, la lingua, l'approccio critico e il sistema di pensiero dell'illuminismo italiano.'''

df["formatted"] = df.apply(
    lambda row: f"""<system>
    {SYSTEM_INSTRUCTION}
    </system>
    <instruction>
    {row['generated_prompt']}
    </instruction>
    <response>
    {row['completion']}
    </response>
    """,
    axis=1
)

In [ ]:
# template = "Instruction:\n{instruction}\n\nResponse:\n{response}"
template = "System:\n{SYSTEM_INSTRUCTION}\n\nInstruction:\n{instruction}\n\nResponse:\n{response}"

In [ ]:
data = df['formatted'].tolist()
data[0]

"<system>\n    Sei un intellettuale e filosofo illuminista italiano del XVIII secolo.\nRispondi alle domande utilizzando lo stile, la lingua, l'approccio critico e il sistema di pensiero dell'illuminismo italiano.\n    </system>\n    <instruction>\n    Descrivi il pericolo della pedanteria nel campo della lingua e della letteratura.\n    </instruction>\n    <response>\n    Forse potran col tempo sembrar troppo animosi alcuni tratti contro i puristi di lingua; ma la pedanteria de’ grammatici, che tenderebbe ad estendersi vergognosamente su tutte le produzioni dell’ingegno;\n    </response>\n    "

## Load Model

KerasNLP provides implementations of many popular [model architectures](https://keras.io/api/keras_nlp/models/). In this tutorial, you'll create a model using `GemmaCausalLM`, an end-to-end Gemma model for causal language modeling. A causal language model predicts the next token based on previous tokens.

Create the model using the `from_preset` method:

In [ ]:
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset("gemma2_2b_en")
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2304)        │   2,614,341,888 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     589,824,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,614,341,888 (9.74 GB)

 Trainable params: 2,614,341,888 (9.74 GB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model_dir = '/content/gdrive/MyDrive/Progetti/fine-tune-gemma/illumigemma/LumiGemma'
gemma_lm = keras_nlp.models.CausalLM.from_preset(model_dir)
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2304)        │   2,614,341,888 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     589,824,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,614,341,888 (9.74 GB)

 Trainable params: 2,614,341,888 (9.74 GB)

 Non-trainable params: 0 (0.00 B)

The `from_preset` method instantiates the model from a preset architecture and weights. In the code above, the string "gemma2_2b_en" specifies the preset architecture — a Gemma model with 2 billion parameters.

NOTE: A Gemma model with 7
billion parameters is also available. To run the larger model in Colab, you need access to the premium GPUs available in paid plans. Alternatively, you can perform [distributed tuning on a Gemma 7B model](https://ai.google.dev/gemma/docs/distributed_tuning) on Kaggle or Google Cloud.

## Inference before fine tuning

In this section, you will query the model with various prompts to see how it responds.

### Prompt sul modello non addestrato

Query the model for suggestions on what to do on a trip to Europe.

In [ ]:
# prompt = template.format(
#     instruction="Scrivi un paragrafo sull'economia pubblica secondo le idee dell'illuminismo italiano. Usa la lingua degli illuministi italiani del '700",
#     # instruction='Can you speak italian?',
#     response="",
# )
# sampler = keras_nlp.samplers.TopKSampler(k=5, seed=2)
# gemma_lm.compile(sampler=sampler)
# # print(gemma_lm.generate(prompt, max_length=256))

# generated_text = gemma_lm.generate(prompt, max_length=256)
# print(generated_text)

In [ ]:
prompt = template.format(
    SYSTEM_INSTRUCTION=SYSTEM_INSTRUCTION,
    instruction="Quale dovrebbe essere il ruolo di un politico nella gestione del commercio di una nazione?",
    response="",
)
sampler = keras_nlp.samplers.TopKSampler(k=5, seed=2)
gemma_lm.compile(sampler=sampler)
print(gemma_lm.generate(prompt, max_length=516))

System:
Sei un intellettuale e filosofo illuminista italiano del XVIII secolo.
Rispondi alle domande utilizzando lo stile, la lingua, l'approccio critico e il sistema di pensiero dell'illuminismo italiano.

Instruction:
Quale dovrebbe essere il ruolo di un politico nella gestione del commercio di una nazione?

Response:
I politici devono fare il loro meglio per assicurare che il commercio sia gestito nel modo più efficiente possibile. Ciò significa che devono essere in grado di gestire la domanda e l'offerta dei beni e servizi, così come le condizioni di mercato.
In generale, un politico dovrebbe promuovere un sistema di commercio basato sul libero scambio e sulla concorrenza. Questo significa che i prezzi del mercato dovrebbero essere liberi e le leggi che limitano la libertà di mercato dovrebbero essere ridotte o eliminate.
Un politico dovrebbe anche essere in grado di garantire la sicurezza dei commercianti e dei viaggiatori. Questo può essere fatto promuovendo un ordine pubblico e 

## LoRA Fine-tuning

To get better responses from the model, fine-tune the model with Low Rank Adaptation (LoRA) using the Databricks Dolly 15k dataset.

The LoRA rank determines the dimensionality of the trainable matrices that are added to the original weights of the LLM. It controls the expressiveness and precision of the fine-tuning adjustments.

A higher rank means more detailed changes are possible, but also means more trainable parameters. A lower rank means less computational overhead, but potentially less precise adaptation.

This tutorial uses a LoRA rank of 4. In practice, begin with a relatively small rank (such as 4, 8, 16). This is computationally efficient for experimentation. Train your model with this rank and evaluate the performance improvement on your task. Gradually increase the rank in subsequent trials and see if that further boosts performance.

In [ ]:
# Enable LoRA for the model and set the LoRA rank to 4.
gemma_lm.backbone.enable_lora(rank=4)
gemma_lm.summary()

Preprocessor: "gemma_causal_lm_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ gemma_tokenizer (GemmaTokenizer)                              │                      Vocab size: 256,000 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2304)        │   2,617,270,528 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     589,824,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,617,270,528 (9.75 GB)

 Trainable params: 2,928,640 (11.17 MB)

 Non-trainable params: 2,614,341,888 (9.74 GB)

Note that enabling LoRA reduces the number of trainable parameters significantly (from 2.6 billion to 2.9 million).

In [ ]:
# Limit the input sequence length to 256 (to control memory usage).
gemma_lm.preprocessor.sequence_length = 256
# Use AdamW (a common optimizer for transformer models).
optimizer = keras.optimizers.AdamW(
    learning_rate=5e-5,
    weight_decay=0.01,
)
# Exclude layernorm and bias terms from decay.
optimizer.exclude_from_weight_decay(var_names=["bias", "scale"])

gemma_lm.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=optimizer,
    weighted_metrics=[keras.metrics.SparseCategoricalAccuracy()],
)
gemma_lm.fit(data, epochs=20, batch_size=16)

Epoch 1/20
474/474 ━━━━━━━━━━━━━━━━━━━━ 353s 625ms/step - loss: 1.1830 - sparse_categorical_accuracy: 0.6766
Epoch 2/20
474/474 ━━━━━━━━━━━━━━━━━━━━ 253s 492ms/step - loss: 1.1740 - sparse_categorical_accuracy: 0.6784
Epoch 3/20
474/474 ━━━━━━━━━━━━━━━━━━━━ 234s 493ms/step - loss: 1.1702 - sparse_categorical_accuracy: 0.6791
Epoch 4/20
474/474 ━━━━━━━━━━━━━━━━━━━━ 234s 493ms/step - loss: 1.1656 - sparse_categorical_accuracy: 0.6799
Epoch 5/20
474/474 ━━━━━━━━━━━━━━━━━━━━ 234s 492ms/step - loss: 1.1603 - sparse_categorical_accuracy: 0.6808
Epoch 6/20
474/474 ━━━━━━━━━━━━━━━━━━━━ 233s 492ms/step - loss: 1.1544 - sparse_categorical_accuracy: 0.6817
Epoch 7/20
474/474 ━━━━━━━━━━━━━━━━━━━━ 234s 493ms/step - loss: 1.1480 - sparse_categorical_accuracy: 0.6831
Epoch 8/20
474/474 ━━━━━━━━━━━━━━━━━━━━ 233s 492ms/step - loss: 1.1413 - sparse_categorical_accuracy: 0.6844
Epoch 9/20
474/474 ━━━━━━━━━━━━━━━━━━━━ 234s 493ms/step - loss: 1.1345 - sparse_categorical_accuracy: 0.6857
Epoch 10/20
474/474

### Note on mixed precision fine-tuning on NVIDIA GPUs

Full precision is recommended for fine-tuning. When fine-tuning on NVIDIA GPUs, note that you can use mixed precision (`keras.mixed_precision.set_global_policy('mixed_bfloat16')`) to speed up training with minimal effect on training quality. Mixed precision fine-tuning does consume more memory so is useful only on larger GPUs.


For inference, half-precision (`keras.config.set_floatx("bfloat16")`) will work and save memory while mixed precision is not applicable.

In [ ]:
# Uncomment the line below if you want to enable mixed precision training on GPUs
# keras.mixed_precision.set_global_policy('mixed_bfloat16')

## Inference after fine-tuning
After fine-tuning, responses follow the instruction provided in the prompt.

### Illuminismo Prompt fine tuned

In [ ]:
prompt = template.format(
    SYSTEM_INSTRUCTION=SYSTEM_INSTRUCTION,
    instruction='''Come dovrebbe essere una città ideale?
    Esponi dettagliatamente le caratteristiche di una città vivibile, progettata a misura dei propri cittadini. ''',
    response="",
)
sampler = keras_nlp.samplers.TopKSampler(k=10, seed=2)
gemma_lm.compile(sampler=sampler)
print(gemma_lm.generate(prompt, max_length=1024))


System:
Sei un intellettuale e filosofo illuminista italiano del XVIII secolo.
Rispondi alle domande utilizzando lo stile, la lingua, l'approccio critico e il sistema di pensiero dell'illuminismo italiano.

Instruction:
Come dovrebbe essere una città ideale?
    Esponi dettagliatamente le caratteristiche di una città vivibile, progettata a misura dei propri cittadini. 

Response:
Una Città ideale non potrà essere, se non in un luogo che si possa raggiungere in breve, e che si trovi a portata de’ cittadini, e che sia in luogo comodo alle arti ed alle imprese de’ medesimi. La più perfetta città dovrà essere un punto di mira di ogni uomo; e la perfezione di questa dovrà essere la massima.


Una città che non sia per lo più una piazza, e una piazza che non sia il luogo in cui si debba trascorrere la maggior parte del tempo, deve essere una città.


## Save fine tuned model

https://www.kaggle.com/code/roumaissaa/offline-finetune-save-load-infer-gemma2b-kerasnlp

Salvataggio nel formato Keras:
https://www.tensorflow.org/tutorials/keras/save_and_load

Kaggle notebook:
https://www.kaggle.com/code/bebechien/how-to-finetuning-gemma2-for-spoken-language-tasks

### Salvataggio del modello standalone e upload su Kaggle

In [ ]:
preset_dir = "/content/gdrive/MyDrive/Progetti/fine-tune-gemma/illumigemma/GemmaEnlightened_v3"
gemma_lm.save_to_preset(preset_dir)

In [ ]:
# Upload the preset as a new model variant on Kaggle
kaggle_uri = "kaggle://stefanotrinchero/lumigemma/keras/gemma2-enlightened_v2"
keras_nlp.upload_preset(kaggle_uri, preset_dir)

Uploading Model https://www.kaggle.com/models/stefanotrinchero/lumigemma/keras/gemma2-enlightened ...
Starting upload for file /content/gdrive/MyDrive/Progetti/fine-tune-gemma/illumigemma/GemmaEnlightened_v3/task.json


Uploading: 100%|██████████| 2.98k/2.98k [00:00<00:00, 3.68kB/s]

Upload successful: /content/gdrive/MyDrive/Progetti/fine-tune-gemma/illumigemma/GemmaEnlightened_v3/task.json (3KB)
Starting upload for file /content/gdrive/MyDrive/Progetti/fine-tune-gemma/illumigemma/GemmaEnlightened_v3/config.json



Uploading: 100%|██████████| 782/782 [00:00<00:00, 1.02kB/s]

Upload successful: /content/gdrive/MyDrive/Progetti/fine-tune-gemma/illumigemma/GemmaEnlightened_v3/config.json (782B)
Starting upload for file /content/gdrive/MyDrive/Progetti/fine-tune-gemma/illumigemma/GemmaEnlightened_v3/model.weights.h5


...


Uploading: 100%|██████████| 591/591 [00:01<00:00, 345B/s]

Upload successful: /content/gdrive/MyDrive/Progetti/fine-tune-gemma/illumigemma/GemmaEnlightened_v3/tokenizer.json (591B)
Starting upload for file /content/gdrive/MyDrive/Progetti/fine-tune-gemma/illumigemma/GemmaEnlightened_v3/assets/tokenizer/vocabulary.spm



Uploading: 100%|██████████| 4.24M/4.24M [00:03<00:00, 1.28MB/s]

Upload successful: /content/gdrive/MyDrive/Progetti/fine-tune-gemma/illumigemma/GemmaEnlightened_v3/assets/tokenizer/vocabulary.spm (4MB)


Your model instance version has been created.
Files are being processed...
See at: https://www.kaggle.com/models/stefanotrinchero/lumigemma/keras/gemma2-enlightened


### Caricamento del modello da Kaggle

In [ ]:
kaggle_uri = "kaggle://stefanotrinchero/lumigemma/keras/gemma2-enlightened"

gemma_lm = keras_nlp.models.CausalLM.from_preset(kaggle_uri)


100%|██████████| 782/782 [00:00<00:00, 1.20MB/s]


100%|██████████| 2.91k/2.91k [00:00<00:00, 4.12MB/s]


100%|██████████| 4.04M/4.04M [00:01<00:00, 2.36MB/s]


100%|██████████| 9.74G/9.74G [11:11<00:00, 15.6MB/s]


### Inference col modello caricato da Kaggle

In [ ]:
SYSTEM_INSTRUCTION = '''Sei un intellettuale e filosofo illuminista italiano del XVIII secolo.
Rispondi alle domande utilizzando lo stile, la lingua, l'approccio critico e il sistema di pensiero dell'illuminismo italiano.'''

prompt = template.format(
    SYSTEM_INSTRUCTION=SYSTEM_INSTRUCTION,
    instruction="Quale dovrebbe essere il ruolo di un politico nella gestione del commercio di una nazione?",
    response="",
)
sampler = keras_nlp.samplers.TopKSampler(k=5, seed=2)
gemma_lm.compile(sampler=sampler)
print(gemma_lm.generate(prompt, max_length=516))

System:
Sei un intellettuale e filosofo illuminista italiano del XVIII secolo.
Rispondi alle domande utilizzando lo stile, la lingua, l'approccio critico e il sistema di pensiero dell'illuminismo italiano.

Instruction:
Quale dovrebbe essere il ruolo di un politico nella gestione del commercio di una nazione?

Response:
A


### Checkpoints

In [ ]:
weights_path = "/content/gdrive/MyDrive/Progetti/fine-tune-gemma/illumigemma/illumigemma.lora.h5"

In [ ]:
gemma_lm.backbone.save_lora_weights("/content/gdrive/MyDrive/Progetti/fine-tune-gemma/illumigemma/lumigemma_v2.lora.h5")

Now that saving the weights is done, we will reset all variables in the current namespace to clean up memory before moving on to the next step.

In [ ]:
%reset -f


## Load fine tuned model

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

Mounted at /content/gdrive


In [ ]:
# Import the keras_nlp library.
import keras_nlp

# Define path to the saved LoRA weights.

LoRA_weights_path = "/content/gdrive/MyDrive/Progetti/fine-tune-gemma/illumigemma/lumigemma_v2.lora.h5"

# Get the GemmaCausalLM model from KerasNLP with specified preset.
model = keras_nlp.models.GemmaCausalLM.from_preset('gemma2_2b_en')

# Enable LoRA rank and set it to 4 for efficient processing (same as before).
model.backbone.enable_lora(rank=4)

# Limit the input sequence length to 512 to control memory usage (same as before).
model.preprocessor.sequence_length = 256

# Load only the LoRA weights.
model.backbone.load_lora_weights(LoRA_weights_path)




In [ ]:
import h5py

# Open the weights file and inspect its contents
with h5py.File(LoRA_weights_path, 'r') as f:
    print(list(f.keys()))

['lora']


Dimensione dei pesi:

In [ ]:
import os

LoRA_weights_path = '/content/gdrive/MyDrive/Progetti/fine-tune-gemma/illumigemma/illumigemma.lora.h5'

print(f"File size: {os.path.getsize(LoRA_weights_path) / (1024 ** 2):.2f} MB")


File size: 11.31 MB


In [ ]:
import h5py

# Inspect the contents of the .h5 file
with h5py.File(LoRA_weights_path, "r") as f:
    print("Keys in the H5 file:")
    print(list(f.keys()))

Keys in the H5 file:
['lora']


In [ ]:
import h5py

# Inspect the 'lora' key
with h5py.File(LoRA_weights_path, "r") as f:
    print("Subkeys in 'lora':")
    print(list(f['lora'].keys()))

Subkeys in 'lora':
['100', '102', '11', '113', '115', '126', '128', '139', '141', '152', '154', '165', '167', '178', '180', '191', '193', '204', '206', '217', '219', '22', '230', '232', '24', '243', '245', '256', '258', '269', '271', '282', '284', '295', '297', '308', '310', '321', '323', '334', '336', '35', '37', '48', '50', '61', '63', '74', '76', '87', '89', '9', 'vars']


In [ ]:
import keras_nlp
print(keras_nlp.__version__)

0.18.1


In [ ]:
print(keras.__version__)

3.7.0


## Summary and next steps

This tutorial covered LoRA fine-tuning on a Gemma model using KerasNLP. Check out the following docs next:

* Learn how to [generate text with a Gemma model](https://ai.google.dev/gemma/docs/get_started).
* Learn how to perform [distributed fine-tuning and inference on a Gemma model](https://ai.google.dev/gemma/docs/distributed_tuning).
* Learn how to [use Gemma open models with Vertex AI](https://cloud.google.com/vertex-ai/docs/generative-ai/open-models/use-gemma).
* Learn how to [fine-tune Gemma using KerasNLP and deploy to Vertex AI](https://github.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/community/model_garden/model_garden_gemma_kerasnlp_to_vertexai.ipynb).

Salvare il modello su Kaggle:

https://www.kaggle.com/code/gpreda/fine-tuning-gemma-2-model-using-lora-and-keras

In [ ]:
preset_dir = ".\gemma2_2b_en_kaggle_docs"
gemma_causal_lm.save_to_preset(preset_dir)

In [ ]:
kaggle_username = os.environ["KAGGLE_USERNAME"]

kaggle_uri = f"kaggle://{kaggle_username}/gemma2-kaggle-docs/keras/gemma2_2b_en_kaggle_docs"
keras_nlp.upload_preset(kaggle_uri, preset_dir)